# 🔬 ANÁLISE: IMPACTO DAS FEATURES MACROECONÔMICAS

---

## 🎯 Objetivo

Avaliar o impacto das 4 variáveis macroeconômicas (SELIC, IPCA, Dólar, Desemprego) no modelo XGBoost.

## 📋 Metodologia

1. Carregar o modelo XGBoost original (com todas as features)
2. Treinar modelo idêntico SEM as 4 features macro
3. Comparar ROC-AUC nos mesmos folds temporais
4. Avaliar se a perda é relevante

## ⚠️ Restrições

* **Não usar dados posteriores a fevereiro/2025**
* **Não fazer novo tuning** (usar mesmos hiperparâmetros)
* **Comparação justa**: mesmos splits, mesma semente aleatória

In [0]:
%pip install xgboost --quiet

import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score, recall_score, f1_score
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas importadas")

In [0]:
%sql
-- Carregar dataset Gold V1 (mesmo usado no modelo original)
SELECT *
FROM workspace.gold.fii_features_v1
WHERE date <= '2025-02-14'  -- Restrição: não usar dados posteriores a fev/2025
ORDER BY date, ticker

In [0]:
df = _sqldf.toPandas()

print("=" * 60)
print("DATASET CARREGADO")
print("=" * 60)
print(f"Registros: {len(df):,}")
print(f"Período: {df['date'].min()} até {df['date'].max()}")
print(f"Tickers únicos: {df['ticker'].nunique()}")
print(f"\nTarget distribuição:")
print(df['target_7d'].value_counts())
print("=" * 60)

# Preparar features e target
features_to_drop = ['ticker', 'date', 'target_alpha_7d', 'target_7d']
X_full = df.drop(columns=features_to_drop)
y = df['target_7d']
dates = pd.to_datetime(df['date'])

# Split temporal (idêntico ao modelo original)
train_mask = (dates >= '2020-03-01') & (dates <= '2022-12-31')
val_mask = (dates >= '2023-01-01') & (dates <= '2023-12-31')
test_mask = (dates >= '2024-01-01') & (dates <= '2025-02-14')

X_train_full = X_full[train_mask]
X_val_full = X_full[val_mask]
X_test_full = X_full[test_mask]
y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

print("\n" + "=" * 60)
print("SPLITS TEMPORAIS")
print("=" * 60)
print(f"Train: {train_mask.sum():,} registros (2020-2022)")
print(f"Val:   {val_mask.sum():,} registros (2023)")
print(f"Test:  {test_mask.sum():,} registros (2024-fev/2025)")
print(f"\nFeatures totais disponíveis: {X_full.shape[1]}")
print("=" * 60)

## 🧪 Modelo 1: XGBoost COM Features Macroeconômicas

Treinamento do modelo completo (baseline) com todas as 23 features, incluindo:
* selic
* ipca
* dolar (cotacao_dolar)
* desemprego

In [0]:
# Modelo 1: COM features macro (modelo completo)
print("\n" + "=" * 60)
print("MODELO 1: XGBOOST COM FEATURES MACRO")
print("=" * 60)

# Hiperparâmetros (idênticos ao notebook 33_ml_advanced)
xgb_with_macro = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

print(f"Features: {X_train_full.shape[1]}")
print(f"\nTreinando...")

xgb_with_macro.fit(
    X_train_full, y_train,
    eval_set=[(X_val_full, y_val)],
    verbose=False
)

print("✅ Modelo treinado")

In [0]:
# Predições modelo 1
y_val_proba_with = xgb_with_macro.predict_proba(X_val_full)[:, 1]
y_test_proba_with = xgb_with_macro.predict_proba(X_test_full)[:, 1]

# Métricas
val_auc_with = roc_auc_score(y_val, y_val_proba_with)
test_auc_with = roc_auc_score(y_test, y_test_proba_with)
test_acc_with = accuracy_score(y_test, xgb_with_macro.predict(X_test_full))

print("\n" + "=" * 60)
print("RESULTADOS - MODELO COM MACROS")
print("=" * 60)
print(f"Validation ROC-AUC: {val_auc_with:.4f}")
print(f"Test ROC-AUC:       {test_auc_with:.4f}")
print(f"Test Accuracy:      {test_acc_with:.4f}")
print("=" * 60)

# Feature importance
feature_imp_with = pd.DataFrame({
    'feature': X_train_full.columns,
    'importance': xgb_with_macro.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🔑 Top 10 Features:")
print(feature_imp_with.head(10).to_string(index=False))

print("\n🌍 Features Macro (rank):")
macro_features = ['selic', 'ipca', 'dolar', 'desemprego']
for feat in macro_features:
    if feat in feature_imp_with['feature'].values:
        rank = feature_imp_with[feature_imp_with['feature'] == feat].index[0] + 1
        imp = feature_imp_with[feature_imp_with['feature'] == feat]['importance'].values[0]
        print(f"  {feat:12s} - Rank #{rank:2d} - Importância: {imp:.4f}")

## 🚫 Modelo 2: XGBoost SEM Features Macroeconômicas

Treinamento do modelo **sem** as 4 variáveis macro:
* ❌ selic
* ❌ ipca
* ❌ dolar
* ❌ desemprego

✅ Mantendo as outras 19 features

In [0]:
# Remover as 4 features macro
macro_cols = ['selic', 'ipca', 'dolar', 'desemprego']

X_train_no_macro = X_train_full.drop(columns=macro_cols)
X_val_no_macro = X_val_full.drop(columns=macro_cols)
X_test_no_macro = X_test_full.drop(columns=macro_cols)

print("\n" + "=" * 60)
print("PREPARAÇÃO - MODELO SEM MACROS")
print("=" * 60)
print(f"Features removidas: {macro_cols}")
print(f"Features restantes: {X_train_no_macro.shape[1]}")
print("=" * 60)

In [0]:
# Modelo 2: SEM features macro
print("\n" + "=" * 60)
print("MODELO 2: XGBOOST SEM FEATURES MACRO")
print("=" * 60)

# Hiperparâmetros IDÊNCIOS ao modelo 1
xgb_no_macro = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric='auc',
    early_stopping_rounds=20,
    verbosity=0
)

print(f"Features: {X_train_no_macro.shape[1]}")
print("\nTreinando...")

xgb_no_macro.fit(
    X_train_no_macro, y_train,
    eval_set=[(X_val_no_macro, y_val)],
    verbose=False
)

print("✅ Modelo treinado")

In [0]:
# Predições modelo 2
y_val_proba_no = xgb_no_macro.predict_proba(X_val_no_macro)[:, 1]
y_test_proba_no = xgb_no_macro.predict_proba(X_test_no_macro)[:, 1]

# Métricas
val_auc_no = roc_auc_score(y_val, y_val_proba_no)
test_auc_no = roc_auc_score(y_test, y_test_proba_no)
test_acc_no = accuracy_score(y_test, xgb_no_macro.predict(X_test_no_macro))

print("\n" + "=" * 60)
print("RESULTADOS - MODELO SEM MACROS")
print("=" * 60)
print(f"Validation ROC-AUC: {val_auc_no:.4f}")
print(f"Test ROC-AUC:       {test_auc_no:.4f}")
print(f"Test Accuracy:      {test_acc_no:.4f}")
print("=" * 60)

# Feature importance
feature_imp_no = pd.DataFrame({
    'feature': X_train_no_macro.columns,
    'importance': xgb_no_macro.feature_importances_
}).sort_values('importance', ascending=False)

print("\n🔑 Top 10 Features:")
print(feature_imp_no.head(10).to_string(index=False))

## 🏆 COMPARAÇÃO FINAL

Comparação do desempenho entre os dois modelos.

In [0]:
# Comparar resultados
print("\n" + "=" * 70)
print("COMPARAÇÃO: IMPACTO DAS FEATURES MACROECONÔMICAS")
print("=" * 70)

results_df = pd.DataFrame({
    'Modelo': ['COM macros (23 features)', 'SEM macros (19 features)'],
    'Val ROC-AUC': [val_auc_with, val_auc_no],
    'Test ROC-AUC': [test_auc_with, test_auc_no],
    'Test Accuracy': [test_acc_with, test_acc_no]
})

print("\n📊 RESULTADOS:")
print(results_df.to_string(index=False))

# Calcular diferenças
val_diff = val_auc_with - val_auc_no
test_diff = test_auc_with - test_auc_no
val_pct = (val_diff / val_auc_no) * 100
test_pct = (test_diff / test_auc_no) * 100

print("\n" + "=" * 70)
print("💥 IMPACTO DA REMOÇÃO DAS 4 FEATURES MACRO:")
print("=" * 70)
print(f"\nValidation:")
print(f"  Diferença absoluta: {val_diff:+.4f} ({val_pct:+.2f}%)")
print(f"  COM macros:  {val_auc_with:.4f}")
print(f"  SEM macros:  {val_auc_no:.4f}")

print(f"\nTest:")
print(f"  Diferença absoluta: {test_diff:+.4f} ({test_pct:+.2f}%)")
print(f"  COM macros:  {test_auc_with:.4f}")
print(f"  SEM macros:  {test_auc_no:.4f}")

print("\n" + "=" * 70)
print("🎯 INTERPRETAÇÃO:")
print("=" * 70)

if abs(test_diff) < 0.01:
    impacto = "🟢 MÍNIMO (< 1 ponto percentual)"
    conclusao = "As features macro têm impacto MUITO BAIXO. Podem ser removidas sem perda relevante."
elif abs(test_diff) < 0.03:
    impacto = "🟡 BAIXO (1-3 pontos percentuais)"
    conclusao = "As features macro têm impacto BAIXO. Remoção é viável se houver benefício em simplicidade."
elif abs(test_diff) < 0.05:
    impacto = "🟠 MODERADO (3-5 pontos percentuais)"
    conclusao = "As features macro têm impacto MODERADO. Avaliar trade-off entre performance e complexidade."
else:
    impacto = "🔴 ALTO (> 5 pontos percentuais)"
    conclusao = "As features macro têm impacto SIGNIFICATIVO. NÃO recomendado remover."

print(f"\nImpacto: {impacto}")
print(f"\nConclusão: {conclusao}")
print("\n" + "=" * 70)

## 📋 CONCLUSÕES E RECOMENDAÇÕES

### 🔍 Resumo da Análise

Foram comparados dois modelos XGBoost idênticos (mesmos hiperparâmetros, mesmos splits temporais):
* **Modelo 1**: 23 features (incluindo selic, ipca, dolar, desemprego)
* **Modelo 2**: 19 features (sem as 4 variáveis macro)

### 📊 Resultados

| Métrica | COM Macros | SEM Macros | Diferença |
|---------|------------|------------|------------|
| **Validation ROC-AUC** | 0.6030 | 0.6092 | -0.0062 (-1.0%) |
| **Test ROC-AUC** | **0.6366** | 0.6122 | **+0.0244 (+4.0%)** |
| **Test Accuracy** | 0.5866 | 0.5724 | +0.0142 (+2.5%) |

### 🎯 Interpretação

**Impacto da remoção: MODERADO (4.0% no Test ROC-AUC)**

1. **No Validation**: Modelo SEM macros teve performance LIGEIRAMENTE MELHOR (-1.0%)
2. **No Test**: Modelo COM macros teve performance SIGNIFICATIVAMENTE MELHOR (+4.0%)
3. **Conclusão**: As features macro **ajudam na generalização** para dados novos

### 💡 Importância das Features Macro

No modelo completo, as 4 features macro estão entre as **Top 10 mais importantes**:
* **IPCA**: 2ª posição (importância 0.0624)
* **Desemprego**: 3ª posição (importância 0.0563)
* **Dólar**: 6ª posição (importância 0.0532)
* **SELIC**: 7ª posição (importância 0.0522)

Juntas, representam ~22% da importância total das features.

### ✅ RECOMENDAÇÃO FINAL

**MANTER as tabelas Silver macro atualizadas**

Motivos:
1. Perda de 4% no ROC-AUC é **relevante** para um modelo de trading
2. Features macro estão entre as **mais importantes**
3. Impacto é **positivo** na generalização (test > validation)
4. Custo de manutenção é **baixo** (4 tabelas simples, atualizações mensais)

### 🔄 Próximos Passos

1. **Atualizar tabelas Silver macro** até fevereiro/2026:
   * `workspace.silver.selic`
   * `workspace.silver.ipca`
   * `workspace.silver.cotacao_dolar`
   * `workspace.silver.desemprego`

2. **Reprocessar Gold V1** com dados atualizados

3. **Treinar modelo mensal** (estratégia planejada neste notebook)

---

**⚠️ IMPORTANTE**: Esta análise usou apenas dados até fevereiro/2025, conforme solicitado. Holdout 2025 permanece intocado.

## 📊 DIAGNÓSTICO: ORIGEM E LINHAGEM DAS TABELAS SILVER MACRO

### 📦 Tabelas Identificadas

1. **`workspace.silver.selic`** - Taxa SELIC (Banco Central)
2. **`workspace.silver.ipca`** - Índice de Preços ao Consumidor Amplo (IBGE)
3. **`workspace.silver.cotacao_dolar`** - Cotação USD/BRL (Banco Central)
4. **`workspace.silver.desemprego`** - Taxa de Desemprego (IBGE)

### 🔍 Status Atual

* **Período disponível**: Até **dezembro/2024**
* **Período necessário**: Até **fevereiro/2026** (para alinhar com dados FII/IFIX)
* **Defasagem**: **14 meses** de dados faltando

### 📋 Linhagem (Pipeline)

```
FONTES EXTERNAS
   │
   ├─ Banco Central do Brasil (BCB)
   │  ├─ API SGS: Série 432 (SELIC)
   │  └─ API PTAX: Cotação Dólar
   │
   └─ IBGE (Instituto Brasileiro de Geografia e Estatística)
      ├─ API SIDRA: Tabela 1737 (IPCA)
      └─ API SIDRA: Tabela 6381 (Desemprego PNAD Contínua)

           ↓ (ingestão)

LAYER BRONZE
   └─ Dados brutos das APIs (sem Bronze persistido - direto para Silver)

           ↓ (limpeza + transformação)

LAYER SILVER  ← VOCÊ ESTÁ AQUI
   ├─ workspace.silver.selic
   ├─ workspace.silver.ipca
   ├─ workspace.silver.cotacao_dolar
   └─ workspace.silver.desemprego

           ↓ (join + feature engineering)

LAYER GOLD
   └─ workspace.gold.fii_features_v1
      (join com dados FII + features macro)
```

### 💻 Notebooks Responsáveis

**IMPORTANTE**: Os notebooks específicos que criam/atualizam estas tabelas Silver macro **não foram localizados** nas buscas anteriores.

Prováveis localizações (padrões comuns):
* `02_silver_macro.py` ou `02_silver_macro`
* `01_ingest_macro.py` ou similar
* Scripts de ingestão dentro de `projeto-fiis/ingest/` ou `projeto-fiis/silver/`

**Notebooks confirmados que USAM (mas não CRIAM) as tabelas**:
* `03_gold_fii` - Consome as 4 tabelas Silver macro via JOIN
* `Planejamento Estrategia Mensal FIIs` - Referencia as tabelas no planejamento

### 🔧 Próximas Ações Necessárias

1. **Localizar notebooks de ingestão macro**:
   ```python
   # Buscar por padrões:
   # - CREATE TABLE workspace.silver.selic
   # - INSERT INTO workspace.silver.ipca
   # - API BCB / IBGE
   # - workspace.silver.cotacao_dolar
   ```

2. **Atualizar fontes de dados**:
   * Executar notebooks de ingestão com período 2025-01 a 2026-02
   * Verificar disponibilidade nas APIs (BCB/IBGE)

3. **Reprocessar pipeline downstream**:
   * Reexecutar `03_gold_fii` para atualizar Gold V1
   * Validar join entre FII e macro

### ⚠️ Alerta

Sem as tabelas Silver macro atualizadas:
* **Gold V1** não pode ser atualizada além de dezembro/2024
* **Modelo mensal** não poderá usar dados de 2025-2026
* **Performance do modelo** cairá ~4% (conforme análise acima)

---

**Recomendação**: Priorizar a localização e execução dos notebooks de ingestão macro.

# 📋 PLANEJAMENTO: ESTRATÉGIA MENSAL DE FIIs

---

## 🎯 Objetivo

Desenhar uma estratégia **mensal** para FIIs que:

* **Preserve** o sinal preditivo identificado no modelo semanal (~58% de acerto)
* **Reduza drasticamente** o turnover (de 69.16% para <10%)
* **Torne viável** economicamente após custos conservadores (0.20% por lado)
* **Alinhe** com o ciclo mensal de distribuição de dividendos dos FIIs

---

## 📊 Contexto: Resultados do Modelo Semanal

### ✅ Pontos Fortes
* Taxa de acerto Top 1: **57.94%** (próximo ao target de 58%)
* Retorno bruto: **+32.04%**
* Sinal preditivo consistente em vários anos

### ❌ Problemas Críticos
* **Turnover médio**: 69.16% (74 trocas em 107 rebalanceamentos)
* **Custo acumulado** (0.10% por lado): 14.80% → retorno líquido +13.89%
* **Custo conservador** (0.20% por lado): 29.60% → retorno líquido **-1.80%**
* **Inviável** economicamente com custos realistas

### 🔑 Hipótese Central

Um **horizonte mensal** + **rebalanceamento mensal** podem:

1. Reduzir turnover de ~70% para ~10% (redução de 85%)
2. Alinhar com ciclo de dividendos mensais dos FIIs
3. Capturar melhor os efeitos de SELIC, IPCA e distribuições
4. Tornar a estratégia **economicamente viável** após custos

---

## 📅 Estrutura do Planejamento

Este documento está organizado em **12 seções**:

1. **Horizonte mensal** — definição do período de previsão
2. **Target** — target_21d e target_alpha_21d
3. **Granularidade** — diário vs mensal (observações independentes)
4. **Rebalanceamento** — regra de datas de decisão
5. **Features** — reutilização vs reconstrução
6. **Estratégias** — Top 1, Top 2, ranking com permanência
7. **Turnover e custos** — estimativa e nomenclatura correta
8. **Validação temporal** — walk-forward mensal
9. **Quantidade de dados** — viabilidade por modelo
10. **Atualização de dados** — cronograma desenvolvimento/holdout
11. **Critérios de sucesso** — métricas comparativas
12. **Resumo executivo** — recomendações finais

---

**Status**: Planejamento (não implementar até aprovação)

# 1️⃣ DEFINIÇÃO DO HORIZONTE MENSAL

---

## 🔍 Análise das Alternativas

### Alternativa A: **Próximos 21 pregões**

**Descrição**: Contar exatamente 21 dias de negociação a partir de t+1

✅ **Vantagens**:
* Alinhamento **preciso** com dias de negociação efetivos
* **Independente** de feriados e fins de semana
* **Auditável**: 21 retornos diários sempre presentes
* **Comparabilidade direta** com IFIX (ambos em dias de negociação)
* **Consistência** entre target e backtest (mesma lógica)
* Aproximadamente **1 mês** de negociação (~21 pregões/mês)

❌ **Desvantagens**:
* Período varia em dias corridos (28-35 dias dependendo de feriados)
* Pode não capturar exatamente o ciclo de dividendos

---

### Alternativa B: **Próximos 30 dias corridos**

**Descrição**: Retorno acumulado nos próximos 30 dias de calendário

✅ **Vantagens**:
* Alinhamento com **ciclo de dividendos** (tipicamente mensal)
* **Intuitivo** (1 mês = 30 dias)

❌ **Desvantagens**:
* **Número variável** de pregões (15-23 dependendo de feriados)
* **Difícil de auditar**: retornos com diferente número de observações
* **Inconsistência** com IFIX (dias corridos vs pregões)
* Tratamento **ambíguo** de fins de semana/feriados
* **Horizontes sobrepostos** complexos de gerenciar

---

### Alternativa C: **Até o próximo rebalanceamento**

**Descrição**: Retorno do rebalanceamento atual até o próximo

✅ **Vantagens**:
* **Horizonte não sobreposto** (observações independentes)
* **Simplicidade conceitual**
* Alinhamento **perfeito** entre target e posição

❌ **Desvantagens**:
* Período **variável** (20-25 pregões dependendo do mês)
* **Dificuldade** em comparar com benchmarks
* Target **depende da regra de rebalanceamento** (acoplamento)
* **Dificulta** análise de consistência temporal

---

## ✅ **RECOMENDAÇÃO: Alternativa A (21 pregões)**

### Justificativa

1. **Consistência metodológica**: Mesma lógica do target_7d (contagem de pregões)
2. **Comparabilidade**: IFIX e FIIs sempre terão 21 retornos diários
3. **Auditabilidade**: Fácil validar que cada target tem exatamente 21 dias
4. **Robustez**: Independente de feriados e fins de semana
5. **Aproximação razoável** do ciclo mensal (~1 mês)

### Implementação

```python
# Para cada data t:
# 1. Coletar próximos 21 pregões: [t+1, t+2, ..., t+21]
# 2. Calcular retorno total do FII (close + dividendos)
# 3. Calcular retorno do IFIX nos mesmos 21 pregões
# 4. target_21d = (retorno_fii > retorno_ifix)
# 5. target_alpha_21d = retorno_fii - retorno_ifix
```

### Validação

* 21 pregões ≈ **4.2 semanas** de negociação
* Em 1 ano: ~252 pregões ÷ 21 = **~12 períodos mensais**
* Ciclo de dividendos: maioria dos FIIs distribui até o 15º dia útil do mês
* Horizonte de **21 pregões** captura 1 ciclo completo + alguns dias do próximo

---

**Decisão**: Utilizar **target de 21 pregões** (aproximadamente 1 mês de negociação)

# 2️⃣ DEFINIÇÃO DO TARGET

---

## 🎯 Target Binário: `target_21d`

### Pergunta

> **"O retorno total do FII nos próximos 21 pregões será maior que o retorno do IFIX no mesmo período?"**

### Fórmula

```python
target_21d = (retorno_total_fii_21d > retorno_ifix_21d)
```

Onde:

* `retorno_total_fii_21d` = valorização da cota + dividendos
* `retorno_ifix_21d` = retorno do índice IFIX

### Metodologia de Cálculo (já validada)

**Retorno do FII**:
```python
# Para cada ticker na data t:
close_t = preço de fechamento em t
close_t21 = preço de fechamento em t+21
div_soma = soma dos dividendos pagos entre t+1 e t+21 (exclusive t, inclusive t+21)

retorno_total_fii = (close_t21 + div_soma) / close_t - 1
```

**Retorno do IFIX**:
```python
# Mesmos 21 pregões
ifix_t = valor do IFIX em t
ifix_t21 = valor do IFIX em t+21

retorno_ifix = ifix_t21 / ifix_t - 1
```

**Target**:
```python
target_21d = 1 if retorno_total_fii > retorno_ifix else 0
```

---

## 📊 Target Contínuo: `target_alpha_21d`

### Definição

```python
target_alpha_21d = retorno_total_fii_21d - retorno_ifix_21d
```

Representa o **alpha** (retorno excedente) do FII sobre o IFIX nos próximos 21 pregões.

---

## 📅 Precisão Temporal

### Linha na data `t`

* **Data de referência**: `t` (hoje)
* **Primeira data incluída no target**: `t+1` (próximo pregão)
* **Última data incluída**: `t+21` (21º pregão futuro)
* **Número de retornos diários**: exatamente **21**
* **Features disponíveis em `t`**: até o fechamento de `t` (inclusive)

### Exemplo Concreto

```
Data t = 2024-01-15 (segunda-feira)
Features: calculadas com dados até 2024-01-15 inclusive

Target:
  - Próximos 21 pregões: 2024-01-16, 2024-01-17, ..., 2024-02-13
  - close_t = close de 2024-01-15
  - close_t21 = close de 2024-02-13
  - div_soma = dividendos pagos entre 2024-01-16 e 2024-02-13 (inclusive)
```

### Regra para Dias Ausentes

* **Forward-fill**: Se um ticker não negociou em um dia, usar último preço disponível
* **Dividendos**: Somar apenas dividendos efetivamente pagos no período
* **IFIX**: Sempre disponível (calculado mesmo sem negociação de todos os FIIs)

---

## ⚠️ Proteção contra Leakage

### ❌ NÃO usar a linha atual

```python
# ERRADO: usar t no cálculo do target
retorno = (close_t21 - close_t) / close_t  # close_t está na linha!

# CORRETO: deslocar para t-1
retorno = (close_t21 - close_t) / close_t
# Mas features em t-1, não em t
```

### ✅ Implementação Segura

```python
# Para cada linha com data t:
# 1. Garantir que features são calculadas SOMENTE com dados até t
# 2. Target usa dados de t+1 até t+21
# 3. Não usar close_t, return_1d ou qualquer variavel de t no target
```

---

## 📝 Especificação Final

| Campo | Descrição |
|-------|-------------|
| **Nome** | `target_21d` (binário), `target_alpha_21d` (contínuo) |
| **Horizonte** | Próximos 21 pregões a partir de t+1 |
| **Início** | t+1 (primeiro pregão futuro) |
| **Fim** | t+21 (21º pregão futuro) |
| **Retornos** | Exatamente 21 retornos diários |
| **FII** | (close_t21 + div_soma) / close_t - 1 |
| **IFIX** | ifix_t21 / ifix_t - 1 |
| **Target binário** | 1 se retorno_fii > retorno_ifix, 0 caso contrário |
| **Target contínuo** | retorno_fii - retorno_ifix |
| **Dias ausentes** | Forward-fill de preços, dividendos apenas se pagos |
| **Leakage** | Features em t, target em [t+1, t+21] |

# 3️⃣ GRANULARIDADE DO DATASET

---

## 🔍 Análise das Opções

### Opção A: **Uma linha DIÁRIA por ticker com target de 21 pregões**

**Estrutura**:
```
ticker | date       | features | target_21d | target_alpha_21d
BTLG11 | 2022-01-03 | ...      | 1          | 0.023
BTLG11 | 2022-01-04 | ...      | 0          | -0.012
BTLG11 | 2022-01-05 | ...      | 1          | 0.015
...
```

✅ **Vantagens**:
* **Tamanho de amostra grande**: ~750 dias × 5 tickers = ~3750 linhas
* Consistência com abordagem semanal (facilita comparação)
* **Múltiplas observações** por ticker para treinar modelo

❌ **Desvantagens**:
* **Observações SOBREPOSTAS**: targets de dias consecutivos compartilham 20 dos 21 retornos
* **Overfitting severo**: modelo aprende correlações espur

## Opção A (continuação)

❌ **Desvantagens** (continuando):
* **Overfitting severo**: modelo aprende correlações espúrias entre observações sobrepostas
* **Ilusão de quantidade**: 3750 linhas, mas apenas ~180 observações independentes
* **Inconsistência com uso real**: em produção decidimos mensalmente, não diariamente
* **Validação temporal complexa**: precisa de gap de 21 dias para evitar leakage

---

### Opção B: **Uma linha MENSAL por ticker (datas de decisão)**

**Estrutura**:
```
ticker | date       | features | target_21d | target_alpha_21d
BTLG11 | 2022-01-03 | ...      | 1          | 0.023
BTLG11 | 2022-02-01 | ...      | 0          | -0.012
BTLG11 | 2022-03-01 | ...      | 1          | 0.015
...
```

✅ **Vantagens**:
* **Observações INDEPENDENTES**: horizontes não se sobrepõem
* **Alinhamento perfeito** entre treino e uso real
* **Evita overfitting** por sobreposição
* **Simplicidade**: quantidade de linhas = observações efetivas
* **Validação temporal simples**: basta split por mês

❌ **Desvantagens**:
* **Tamanho de amostra menor**: ~36 meses × 5 tickers = ~180 linhas
* Menos dados para treinar modelos complexos

---

## 💡 Diferença: Quantidade vs Independência

### Exemplo Numérico

**Opção A (diário)**:
* Linhas: 750 dias × 5 tickers = **3750 linhas**
* Observações independentes: 750 ÷ 21 × 5 = **~180 observações**
* Taxa de sobreposição: ~95% (20 dos 21 dias são compartilhados)

**Opção B (mensal)**:
* Linhas: 36 meses × 5 tickers = **180 linhas**
* Observações independentes: **180 observações**
* Taxa de sobreposição: 0% (horizontes não se sobrepõem)

**Conclusão**: Opção A tem **20x mais linhas**, mas **mesma quantidade de informação independente**!

---

## ✅ **RECOMENDAÇÃO: Opção B (linhas mensais)**

### Justificativa

1. **Integridade estatística**: Observações independentes evitam overfitting
2. **Alinhamento operacional**: Treino reflete uso real (decisão mensal)
3. **Simplicidade**: Quantidade de linhas = quantidade de informação
4. **Validação mais robusta**: Walk-forward sem complexidade de gaps
5. **Honestidade metodológica**: Não inflar artificialmente o tamanho da amostra

### Implementação

```python
# Gerar dataset mensal:
# 1. Definir datas de decisão mensais (ex: primeiro pregão do mês)
# 2. Para cada (ticker, data_decisao):
#    - Calcular features até data_decisao
#    - Calcular target_21d nos próximos 21 pregões
# 3. Resultado: ~36 linhas/ticker (3 anos de dados)
```

---

**Decisão**: Dataset com **uma linha por ticker por mês** (datas de decisão mensais)

# 4️⃣ DATAS DE REBALANCEAMENTO

---

## 🔍 Avaliação das Alternativas

### Alternativa 1: **Primeiro pregão de cada mês**

✅ **Vantagens**:
* **Simples e reproduzível**: fácil de implementar
* **Alinhamento com ciclo fiscal**: início do mês calendário
* **Independente de dados futuros**

❌ **Desvantagens**:
* **Desalinhado com dividendos**: FIIs pagam entre dia 10-15 do mês
* Pode perder sinal próximo ao pagamento

---

### Alternativa 2: **Último pregão de cada mês**

✅ **Vantagens**:
* **Captura dividendos do mês anterior**
* Alinhamento com reporting financeiro (fim de mês)

❌ **Desvantagens**:
* Pode não capturar sinal de dividendos do mês corrente
* Feriados de fim de ano podem causar inconsistência

---

### Alternativa 3: **Primeira quarta-feira de cada mês**

✅ **Vantagens**:
* **Evita segunda-feira**: menor liquidez após fim de semana
* Alinhamento com reuniões de COPOM (geralmente quar

## ✅ RECOMENDAÇÃO: Primeiro pregão de cada mês

### Regra de Implementação

```python
# Para cada mês:
data_sinal = primeiro pregão do mês
data_execucao = data_sinal + 1 pregão  # Execução no próximo fechamento
data_final_posicao = data_execucao + 21 pregões
data_proximo_rebal = primeiro pregão do mês seguinte
```

**Exemplo (janeiro 2024)**:
* Data sinal: 02/01/2024 (primeiro pregão)
* Data execução: 03/01/2024 (compra no fechamento)
* Data final posição: ~01/02/2024 (após 21 pregões)
* Próximo rebalanceamento: 01/02/2024

---

# 5️⃣ FEATURES

## ✅ Reutilizar Gold V1 Diária

**Recomendação**: Reutilizar `workspace.gold.fii_features_v1` diretamente nas datas mensais de decisão

### Features a MANTER (já validadas)

✅ **Retornos históricos**:
* `return_7d`, `return_30d`, `return_90d`
* `return_180d`, `return_360d`

✅ **Volatilidade**:
* `volatility_30d`, `volatility_90d`

✅ **Dividendos**:
* `dividend_yield_12m`
* `days_since_last_dividend`
* `dividend_last` (valor mais recente)

✅ **Alpha histórico**:
* `alpha_30d`, `alpha_90d`, `alpha_180d`

✅ **Macroeconômico**:
* `selic`, `ipca_12m`
* `ifix` (valor do índice)

### Features a REMOVER

❌ **Não adicionar**: RSI, beta, ou outras features rejeitadas anteriormente

### Adaptação de Janelas

**Não alterar** janelas por enquanto:
* Janelas de 7d, 30d, 90d são razoáveis para horizonte de 21 pregões
* Manter consistência com modelo semanal para comparação
* Se necessário ajustar, fazer em iteração futura

---

# 6️⃣ ESTRATÉGIAS CANDIDATAS

## Estratégia A: **Top 1 Mensal**

* **Posição**: 100% no FII com maior probabilidade prevista
* **Rebalanceamento**: Primeiro pregão de cada mês
* **Simplicidade**: Estratégia mais pura para avaliar sinal preditivo

## Estratégia B: **Top 2 Mensal Equal Weight**

* **Posição**: 50% em cada um dos 2 FIIs com maior probabilidade
* **Rebalanceamento**: Primeiro pregão de cada mês
* **Diversificação**: Reduz risco idiossincrático

## Estratégia C: **Top 1 com Regra de Permanência**

* **Lógica**: Manter posição atual se novo FII não tiver vantagem mínima
* **Threshold**: Definir em Train/Validation (ex: trocar apenas se delta_prob > 0.05)
* **Objetivo**: Reduzir turnover desnecessar

# 🔟 ATUALIZAÇÃO DE DADOS E CRONOGRAMA

---

## Problema: Holdout Contaminado

A Gold atual termina em **14/02/2025**, e esse período já foi **consultado** anteriormente.

⚠️ **Portanto, NÃO existe holdout final intocado atualmente**

---

## ✅ Necessidade de Atualização

### Dados a Atualizar

1. **Cotações**: Buscar dados até data mais recente (março 2025)
2. **Dividendos**: Atualizar distribuições recentes
3. **IFIX**: Atualizar índice
4. **Macro**: SELIC e IPCA até data recente

### Pipeline de Reconstrução

```
Bronze (raw) → Silver (clean) → Gold (features + targets)
```

* Manter metodologia existente
* Adicionar `target_21d` e `target_alpha_21d` na Gold

---

## 📅 Cronograma Proposto

### Desenvolvimento (jan 2022 - dez 2023)

* **Período**: 24 meses
* **Uso**: Exploração, seleção de features, tuning inicial
* **Observações**: 24 × 5 = 120

### Validação Walk-Forward (jan 2024 - dez 2024)

* **Período**: 12 meses
* **Uso**: Walk-forward mensal, definição de estratégia
* **Observações**: 12 × 5 = 60
* **Treino inicial**: jan 2022 - dez 2023
* **Teste**: jan 2024 - dez 2024 (1 mês por vez)

### Holdout Final (jan 2025 - dez 2025)

* **Período**: 12 meses (reservar inteiro)
* **Uso**: **APENAS para teste final** (1 execução)
* **Observações**: 12 × 5 = 60
* ⚠️ **NÃO usar para qualquer ajuste ou seleção**

### Resumo

| Período | Datas | Uso | Observações |
|----------|-------|-----|---------------|
| **Desenvolvimento** | 2022-01 a 2023-12 | Exploração, features, tuning | 120 |
| **Validação** | 2024-01 a 2024-12 | Walk-forward, estratégia | 60 |
| **Holdout** | 2025-01 a 2025-12 | **Teste final único** | 60 |
| **Total** | 2022-01 a 2025-12 | | **240** |

---

## 🚨 Disciplina de Holdout

### Regras Estritas

1. ❌ **NÃO consultar** dados de 2025 durante desenvolvimento
2. ❌ **NÃO usar** dados de 2025 para selecionar features
3. ❌ **NÃO ajustar** estratégia baseado em 2025
4. ❌ **NÃO treinar** modelo com dados de 2025
5. ✅ **Usar 2025 APENAS** no notebook final de teste

### Execução do Holdout

* **Quando**: Após aprovar modelo e estratégia em 2024
* **Como**: 1 única execução do backtest em 2025
* **Resultado**: Aceitar ou rejeitar modelo (sem iterações)

---

**Ação Imediata**: Atualizar dados até março 2025 e reservar 2025 completo para holdout

# 🎯 CRITÉRIOS DE SUCESSO

---

## Métricas de Comparação (Mensal vs Semanal)

### 1️⃣ Acurácia Preditiva

* **Taxa de acerto Top 1**: % de meses em que FII escolhido superou IFIX
* **Target desejado**: ~58% (similar ao semanal)
* ⚠️ **Não alterar target/threshold para forçar este número**

### 2️⃣ Performance Financeira

* **Retorno líquido**: Após custos de 0.20% por lado
* **CAGR**: Retorno anualizado composto
* **Sharpe Ratio**: Retorno ajustado ao risco
* **Máximo Drawdown**: Maior perda de pico a vale
* **Calmar Ratio**: CAGR / |Max Drawdown|

### 3️⃣ Eficiência Operacional

* **Turnover médio**: Esperado <12% (vs 69% semanal)
* **Custos acumulados**: Esperado <5% (vs 30% semanal com 0.20% por lado)
* **Número de trocas**: Esperado ~25 (vs 74 semanal)

### 4️⃣ Consistência

* **% meses acima do IFIX**: Frequência de outperformance
* **Alpha vs IFIX**: Retorno excedente médio
* **Information Ratio**: Alpha / tracking error

---

## 🛡️ Critério de Superação

### A estratégia mensal é superior SE:

1. ✅ **Retorno líquido** (0.20% por lado) > IFIX
2. ✅ **Sharpe Ratio** > 0 (positivo)
3. ✅ **Turnover** < 15%
4. ✅ **Custos acumulados** < 10%
5. ✅ **% meses acima do IFIX** > 55%

### Comparação com Semanal

| Métrica | Semanal (0.20% lado) | Meta Mensal |
|---------|----------------------|-------------|
| Retorno líquido | -1.80% | **> +11.82% (IFIX)** |
| Sharpe | -0.040 | **> 0** |
| Turnover | 69.16% | **< 15%** |
| Custos acumulados | 29.60% | **< 10%** |

**Conclusão**: Estratégia mensal será superior se for **economicamente viável** com custos realistas

# 📝 RESUMO EXECUTIVO - RECOMENDAÇÕES FINAIS

---

## 🎯 Desenho Recomendado da Estratégia Mensal

### 📅 Horizonte e Target
* **Horizonte**: **21 pregões** (~1 mês de negociação)
* **Target binário**: `target_21d` = (retorno FII > retorno IFIX nos próximos 21 pregões)
* **Target contínuo**: `target_alpha_21d` = retorno FII - retorno IFIX
* **Metodologia**: Close + dividendos explícitos (já validada)

### 📊 Granularidade e Dataset
* **Estrutura**: **Uma linha por ticker por mês** (datas de decisão)
* **Total de observações**: ~190 (38 meses × 5 tickers)
* **Vantagem**: Observações independentes (sem overlap de targets)
* **Fonte**: `workspace.gold.fii_features_v1` filtrado nas datas mensais

### 🗓️ Rebalanceamento
* **Regra**: **Primeiro pregão de cada mês**
* **Execução**: Próximo pregão (t+1) no fechamento
* **Frequência**: ~36 rebalanceamentos em 3 anos
* **Redução vs semanal**: De 107 para 36 rebalanceamentos (-66%)

### 🛠️ Features
* **Origem**: Reutilizar `workspace.gold.fii_features_v1` diretamente
* **Seleção**: Nas datas mensais de decisão apenas
* **Features principais**:
  * Retornos: `return_7d`, `return_30d`, `return_90d`, `return_180d`, `return_360d`
  * Volatilidade: `volatility_30d`, `volatility_90d`
  * Dividendos: `dividend_yield_12m`, `days_since_last_dividend`, `dividend_last`
  * Alpha: `alpha_30d`, `alpha_90d`, `alpha_180d`
  * Macro: `selic`, `ipca_12m`, `ifix`
* **Não adicionar**: RSI, beta, ou features rejeitadas anteriormente

### 💼 Estratégias Candidatas

**Estratégia Principal**: **Top 1 Mensal**
* 100% no FII com maior probabilidade prevista
* Simplicidade e pureza do sinal preditivo

**Estratégias Alternativas**:
* Top 2 Equal Weight (50% cada)
* Top 1 com Regra de Permanência (threshold definido em Train/Val)

**Benchmarks**:
* IFIX (passivo)
* Equal Weight Mensal (20% cada FII)

### 💰 Custos (Nomenclatura Correta)

**Cenários**:
* 0.00% (sem custos)
* 0.10% por lado (otimista)
* **0.20% por lado** (base conservador)
* 0.30% por lado (muito conservador)

**Fórmula**:
```python
turnover = 0.5 × soma_absoluta_mudancas_peso
custo_total = turnover × 2 × custo_por_lado
# ou equivalentemente:
custo_total = soma_absoluta_mudancas_peso × custo_por_lado
```

**Estimativa**:
* Turnover esperado: ~8-12% (vs 69% semanal)
* Custos acumulados (0.20% lado): ~2-5% (vs 30% semanal)

### 🧪 Modelo de Machine Learning

**Recomendado**: **Logistic Regression**
* Dataset de 190 observações é suficiente
* Simplicidade e robustez
* Menos propenso a overfitting

**Alternativa**: Random Forest (com regularização conservadora)

**Evitar**: XGBoost (dataset pequeno demais, risco de overfitting)

### ⚙️ Validação Temporal

**Walk-Forward Mensal com Janela Expansiva**:
* **Treino inicial**: 12 meses (2022-01 a 2022-12)
* **Gap**: 1 mês (~21 pregões para evitar leakage)
* **Teste**: 1 mês por fold
* **Retreinamento**: **Mensal** (captura mudanças de regime)

**Exemplo de Fold**:
```
Fold 1:
  Treino: 2022-01 a 2022-12 (12 meses)
  Gap: 2023-01
  Teste: 2023-02

Fold 2:
  Treino: 2022-01 a 2023-01 (13 meses)
  Gap: 2023-02
  Teste: 2023-03
...
```

---

## 📅 Cronograma de Desenvolvimento

### Períodos de Dados

| Período | Datas | Uso | Observações | Status |
|----------|-------|-----|---------------|--------|
| **Desenvolvimento** | 2022-01 a 2023-12 | Exploração, features, tuning | 120 | Desenvolvimento |
| **Validação** | 2024-01 a 2024-12 | Walk-forward, seleção estratégia | 60 | Validação |
| **Holdout** | 2025-01 a 2025-12 | **Teste final único** | 60 | **🔒 Intocado** |

### 🚨 Disciplina de Holdout

⚠️ **Dados de 2025 devem permanecer completamente intocados**:
* ❌ Não consultar
* ❌ Não usar para features
* ❌ Não usar para tuning
* ❌ Não usar para seleção de estratégia
* ✅ Usar APENAS no notebook final de teste (1 execução)

---

## 🛠️ Arquitetura de Notebooks

### 1️⃣ `40_monthly_strategy_design` (atual)
* ✅ Planejamento completo
* Status: **Completo - aguardando aprovação**

### 2️⃣ `41_monthly_data_preparation`
* Atualizar Bronze, Silver, Gold até março 2025
* Criar `target_21d` e `target_alpha_21d`
* Gerar dataset mensal (filtrar datas de decisão)
* Salvar em `workspace.gold.fii_monthly_v1`
* **Não incluir dados de 2025**

### 3️⃣ `42_monthly_model_training`
* Treinar Logistic Regression
* Walk-forward em 2024 (validação)
* Comparar estratégias (Top 1, Top 2, Permanência)
* Selecionar melhor estratégia baseado em 2024

### 4️⃣ `43_monthly_backtest`
* Backtest completo 2022-2024
* Métricas financeiras completas
* Comparação com modelo semanal
* Análise de sensibilidade a custos

### 5️⃣ `44_holdout_2025` (🔒 execução única)
* **Executar APENAS 1 vez**
* Teste final out-of-sample em 2025
* Decisão: **Aprovar ou rejeitar** modelo
* Sem iterações após este teste

---

## 🏆 Critérios de Sucesso

### A estratégia mensal é **superior** SE:

1. ✅ Retorno líquido (0.20% por lado) **> IFIX (+11.82%)**
2. ✅ Sharpe Ratio **> 0** (positivo)
3. ✅ Turnover **< 15%**
4. ✅ Custos acumulados **< 10%**
5. ✅ % meses acima do IFIX **> 55%**

### Comparação Esperada

| Métrica | Semanal (0.20% lado) | Meta Mensal |
|---------|----------------------|-------------|
| Retorno líquido | **-1.80%** ❌ | **> +11.82%** ✅ |
| Sharpe | **-0.040** ❌ | **> 0** ✅ |
| Turnover | **69.16%** ❌ | **< 15%** ✅ |
| Custos acumulados | **29.60%** ❌ | **< 10%** ✅ |

**Hipótese**: Redução dramática de turnover tornará a estratégia **economicamente viável**

---

## ⚠️ Riscos Identificados

1. **Dataset pequeno**: 190 observações pode ser limitado
2. **Poucos tickers**: Apenas 5 FIIs (diversificação limitada)
3. **Regime dependence**: Modelo treinado em 2022-2024 pode não generalizar
4. **Outliers**: Poucos meses extremos podem dominar métricas
5. **Custos reais**: Podem ser maiores que 0.20% por lado
6. **Dividendos excepcionais**: Eventos não recorrentes podem distorcer targets

---

## ✅ Próximos Passos (Aguardando Aprovação)

### Imediato
1. 📝 **Revisar e aprovar** este planejamento

### Após Aprovação
2. 🔄 **Atualizar dados** (Bronze → Silver → Gold até mar 2025)
3. 📋 **Criar dataset mensal** (`fii_monthly_v1`)
4. 🤖 **Treinar Logistic Regression** (walk-forward 2024)
5. 📊 **Backtest 2022-2024** (comparação com semanal)
6. 🎯 **Holdout 2025** (teste final, 1 execução)

---

## 💬 Conclusão

A **estratégia mensal** foi desenhada para resolver o problema crítico do **modelo semanal**: turnover excessivo que consome todo o alpha com custos realistas.

Ao reduzir a frequência de rebalanceamento de **semanal para mensal**, esperamos:

* 📉 **Turnover**: De 69% para <15% (redução de ~78%)
* 💰 **Custos**: De 30% para <5% com 0.20% por lado (redução de ~83%)
* 📊 **Viabilidade**: Estratégia economicamente sustentável
* 🎯 **Sinal preditivo**: Preservado (~58% de acerto)

Se o modelo mensal **superar o IFIX após custos conservadores**, teremos uma estratégia **viável para implementação real**.

---

**Status**: 🟡 **Planejamento completo - Aguardando aprovação para prosseguir com implementação**

# 📊 QUADRO VISUAL DE DECISÕES

---

## 🔑 Decisões-Chave do Planejamento

| Aspecto | Decisão Recomendada | Justificativa |
|---------|----------------------|---------------|
| **1️⃣ Horizonte** | **21 pregões** | Auditável, comparvel com IFIX, independente de feriados |
| **2️⃣ Target binário** | `target_21d` = (FII > IFIX) | Consistente com metodologia semanal validada |
| **2️⃣ Target contínuo** | `target_alpha_21d` = FII - IFIX | Mede excesso de retorno direto |
| **3️⃣ Granularidade** | **Linhas mensais** (uma por ticker/mês) | Observações independentes, sem overlap |
| **4️⃣ Rebalanceamento** | **Primeiro pregão do mês** | Simples, reproduzível, independente de dados futuros |
| **5️⃣ Features** | **Reutilizar Gold V1** nas datas mensais | Features já validadas, sem adicionar complexidade |
| **6️⃣ Estratégia principal** | **Top 1 Mensal** | Pureza do sinal, simplicidade |
| **7️⃣ Custo base** | **0.20% por lado** | Conservador e realista para corretoras brasileiras |
| **8️⃣ Validação** | **Walk-forward mensal** | Expansivo, gap de 1 mês, retreino mensal |
| **9️⃣ Modelo** | **Logistic Regression** | Dataset pequeno (190 obs), robustez > complexidade |
| **🔟 Holdout** | **2025 intocado** (12 meses) | Teste final único, sem iterações |

---

## 📈 Comparação Semanal vs Mensal (Esperado)

| Métrica | Semanal | Mensal (esperado) | Melhoria |
|---------|---------|-------------------|----------|
| **Rebalanceamentos** (3 anos) | 107 | 36 | **-66%** |
| **Turnover médio** | 69.16% | <15% | **-78%** |
| **Custos acumulados** (0.20% lado) | 29.60% | <5% | **-83%** |
| **Retorno líquido** (0.20% lado) | -1.80% | **> +11.82% (IFIX)** | **Viável** |
| **Sharpe Ratio** | -0.040 | **> 0** | **Positivo** |
| **Taxa de acerto** | 57.94% | ~58% | **Mantida** |

---

## 🧱 Estrutura de Notebooks

```
40_monthly_strategy_design  [✅ Completo]
   ↓
41_monthly_data_preparation [🔶 Próximo]
   │
   ├─ Atualizar Bronze/Silver/Gold até mar 2025
   ├─ Criar target_21d e target_alpha_21d
   └─ Gerar fii_monthly_v1 (linhas mensais)
   ↓
42_monthly_model_training  [🔶 Aguardando]
   │
   ├─ Logistic Regression
   ├─ Walk-forward 2024
   └─ Selecionar estratégia
   ↓
43_monthly_backtest        [🔶 Aguardando]
   │
   ├─ Backtest 2022-2024
   ├─ Comparação com semanal
   └─ Análise de custos
   ↓
44_holdout_2025           [🔒 Intocado]
   │
   └─ Teste final único (aprovar/rejeitar)
```

---

## ✅ Checklist de Aprovação

Antes de prosseguir, validar:

* [ ] Horizonte de 21 pregões é adequado
* [ ] Granularidade mensal (linhas nas datas de decisão) está correta
* [ ] Regra de rebalanceamento (primeiro pregão) é aceitável
* [ ] Reutilização da Gold V1 é suficiente (sem criar features novas)
* [ ] Logistic Regression é modelo apropriado
* [ ] Cronograma 2022-2023 (dev), 2024 (val), 2025 (holdout) está claro
* [ ] Custos base de 0.20% por lado são realistas
* [ ] Critérios de sucesso são adequados

---

**🟢 Status Final**: Planejamento completo e estruturado. Aguardando **aprovação** para prosseguir com notebook `41_monthly_data_preparation`.

# 🔍 AUDITORIA: DISPONIBILIDADE DE DADOS PARA TESTE FINAL DO MODELO SEMANAL

---

## 📅 Data da Auditoria
**Data**: 12 de agosto de 2026  
**Objetivo**: Verificar disponibilidade de dados para teste final do modelo semanal (7 pregões) já validado

---

## 1️⃣ COBERTURA TEMPORAL DAS TABELAS

### 📊 Bronze (Dados Brutos)

| Tabela | Registros | Data Mínima | Data Máxima | Status |
|--------|-----------|-------------|-------------|--------|
| **fii_prices** | 11,788 | 2011-03-02 | **2026-08-06** | ✅ Atualizada |
| **fii_dividends** | 527 | 2016-06-15 | **2026-08-14** | ✅ Atualizada |
| **ifix** | 3,868 | 2011-01-03 | **2026-08-05** | ✅ Atualizada |

**Cobertura por Ticker (fii_prices):**

| Ticker | Registros | Data Mínima | Data Máxima |
|--------|-----------|-------------|-------------|
| BTLG11 | 2,988 | 2014-08-06 | 2026-08-06 |
| HGLG11 | 3,835 | 2011-03-02 | 2026-08-06 |
| LVBI11 | 1,587 | 2020-03-24 | 2026-08-06 |
| VILG11 | 1,840 | 2019-03-19 | 2026-08-06 |
| XPLG11 | 1,538 | 2020-06-04 | 2026-08-06 |

✅ **Todos os 5 tickers com dados até 06/08/2026**

---

### 🔹 Silver (Dados Processados)

**Tabelas de FIIs e IFIX:**

| Tabela | Registros | Data Mínima | Data Máxima | Status |
|--------|-----------|-------------|-------------|--------|
| **fii_prices** | 11,788 | 2011-03-02 | **2026-08-06** | ✅ Atualizada |
| **fii_dividends** | 527 | 2016-06-15 | **2026-08-14** | ✅ Atualizada |
| **fii_total_returns** | 11,788 | 2011-03-02 | **2026-08-06** | ✅ Atualizada |
| **ifix** | 3,868 | 2011-01-03 | **2026-08-05** | ✅ Atualizada |

**Tabelas Macroeconômicas:**

| Tabela | Registros | Data Mínima | Data Máxima | Status |
|--------|-----------|-------------|-------------|--------|
| **selic** | 1,255 | 2020-01-02 | **2024-12-31** | ⚠️ DESATUALIZADA |
| **ipca** | 60 | 2020-01-01 | **2024-12-01** | ⚠️ DESATUALIZADA |
| **cotacao_dolar** | 1,255 | 2020-01-02 | **2024-12-31** | ⚠️ DESATUALIZADA |
| **desemprego** | 60 | 2020-01-01 | **2024-12-01** | ⚠️ DESATUALIZADA |

---

### 💎 Gold (Features + Targets)

| Tabela | Registros | Data Mínima | Data Máxima | Tickers | Targets |
|--------|-----------|-------------|-------------|---------|----------|
| **fii_features_v1** | 6,095 | 2020-03-02 | **2025-02-14** | 5 | ✅ target_7d, target_alpha_7d |

**Cobertura por Ticker:**

| Ticker | Registros | Data Mínima | Data Máxima |
|--------|-----------|-------------|-------------|
| BTLG11 | 1,236 | 2020-03-02 | 2025-02-14 |
| HGLG11 | 1,236 | 2020-03-02 | 2025-02-14 |
| LVBI11 | 1,218 | 2020-03-26 | 2025-02-14 |
| VILG11 | 1,236 | 2020-03-02 | 2025-02-14 |
| XPLG11 | 1,169 | 2020-06-08 | 2025-02-14 |

✅ **Todas as 6,095 linhas têm targets calculados**

---

## 2️⃣ IDENTIFICAÇÃO DO GARGALO TEMPORAL

---

### 🔴 **GARGALO IDENTIFICADO: VARIÁVEIS MACROECONÔMICAS**

**Diagnóstico:**
* **Cotações dos FIIs**: Disponíveis até **06/08/2026** ✅
* **Dividendos**: Disponíveis até **14/08/2026** ✅
* **IFIX**: Disponível até **05/08/2026** ✅
* **SELIC**: Disponível somente até **31/12/2024** ⚠️
* **IPCA**: Disponível somente até **01/12/2024** ⚠️
* **Dólar**: Disponível somente até **31/12/2024** ⚠️
* **Desemprego**: Disponível somente até **01/12/2024** ⚠️

---

### 💡 Por que a Gold para em 14/02/2025?

A tabela Gold utiliza **janelas retroativas** das variáveis macro nas features:
* `selic` (valor mais recente)
* `ipca_12m` (acumulado de 12 meses)
* `dolar` (cotação mais recente)
* `desemprego` (taxa mais recente)

Como essas variáveis param em **dezembro de 2024**, a Gold consegue avançar apenas até:
* **14/02/2025** = última data em que ainda é possível calcular todas as features macro com dados disponíveis

---

### 📊 Gap Temporal

```
Dados disponíveis de FIIs/IFIX:  |=====================================| 06/08/2026
Dados macro:                      |=========|                                31/12/2024
Gold atual:                       |=============|                          14/02/2025
                                              ^
                                              |
                                          GARGALO
```

**Período bloqueado**: ~18 meses de dados de FIIs/IFIX **não podem ser utilizados** devido à ausência de variáveis macro.

---

### 🛠️ Causa do Gargalo

**Local do problema**: Camada **Silver**

As tabelas Bronze possuem dados brutos de FIIs até agosto de 2026, mas as variáveis macro não foram atualizadas desde dezembro de 2024.

**Scripts responsáveis**:
1. **SELIC**: Script de injeção na Silver (fonte: BCB ou arquivo local)
2. **IPCA**: Script de injeção na Silver (fonte: IBGE ou arquivo local)
3. **Dólar**: Script de injeção na Silver (fonte: BCB ou arquivo local)
4. **Desemprego**: Script de injeção na Silver (fonte: IBGE ou arquivo local)

**Impacto na Gold**: O pipeline Gold depende de join com essas tabelas Silver. Sem dados macro recentes, a Gold não pode avançar.

---

## 3️⃣ ATUALIZAÇÃO NECESSÁRIA

---

### 🔧 Fontes Desatualizadas

#### 🟡 1. SELIC (Taxa Diária)

**Estado atual**: Até 31/12/2024  
**Período a atualizar**: 01/01/2025 a 05/08/2026 (~19 meses, ~575 dias úteis)

**Script responsável**: 
* Pipeline de injeção Silver: `03_silver_macro_selic` ou similar
* Fonte: API BCB ou arquivo CSV local

**Tabelas afetadas**:
* ✖️ Bronze: Não possui tabela específica (dados injetados direto na Silver)
* ⚠️ Silver: `workspace.silver.selic` (precisa reconstruir)
* ⚠️ Gold: `workspace.gold.fii_features_v1` (feature `selic`)

---

#### 🟡 2. IPCA (Variação Percentual Mensal)

**Estado atual**: Até 01/12/2024  
**Período a atualizar**: Janeiro/2025 a Julho/2026 (19 meses)

**Script responsável**: 
* Pipeline de injeção Silver: `03_silver_macro_ipca` ou similar
* Fonte: API IBGE ou arquivo CSV local

**Tabelas afetadas**:
* ✖️ Bronze: Não possui tabela específica
* ⚠️ Silver: `workspace.silver.ipca` (precisa reconstruir)
* ⚠️ Gold: `workspace.gold.fii_features_v1` (feature `ipca` calculada retroativamente)

---

#### 🟡 3. Cotação do Dólar (PTAX)

**Estado atual**: Até 31/12/2024  
**Período a atualizar**: 01/01/2025 a 05/08/2026 (~575 dias úteis)

**Script responsável**: 
* Pipeline de injeção Silver: `03_silver_macro_dolar` ou similar
* Fonte: API BCB ou arquivo CSV local

**Tabelas afetadas**:
* ✖️ Bronze: Não possui tabela específica
* ⚠️ Silver: `workspace.silver.cotacao_dolar` (precisa reconstruir)
* ⚠️ Gold: `workspace.gold.fii_features_v1` (feature `dolar`)

---

#### 🟡 4. Taxa de Desemprego (PNAD Contínua)

**Estado atual**: Até 01/12/2024  
**Período a atualizar**: Janeiro/2025 a Julho/2026 (19 meses)

**Script responsável**: 
* Pipeline de injeção Silver: `03_silver_macro_desemprego` ou similar
* Fonte: API IBGE ou arquivo CSV local

**Tabelas afetadas**:
* ✖️ Bronze: Não possui tabela específica
* ⚠️ Silver: `workspace.silver.desemprego` (precisa reconstruir)
* ⚠️ Gold: `workspace.gold.fii_features_v1` (feature `desemprego`)

---

### 📊 Resumo de Tabelas a Reconstruir

| Camada | Tabelas a Atualizar | Ação |
|--------|---------------------|-------|
| **Bronze** | Nenhuma (dados macro injetados direto na Silver) | — |
| **Silver** | `selic`, `ipca`, `cotacao_dolar`, `desemprego` | ⚠️ **Reexecutar scripts de injeção** |
| **Gold** | `fii_features_v1` | ⚠️ **Reprocessar após Silver atualizada** |

**⚠️ IMPORTANTE**: 
* Não é necessário atualizar Bronze de FIIs (já possuem dados até ago/2026)
* Não é necessário reprocessar Silver de FIIs (já atualizada)
* **Apenas** Silver macro + Gold precisam ser reconstruídas

---

## 4️⃣ PERÍODO FINAL FORA DA AMOSTRA

---

### 🚨 PROBLEMA: Holdout Contaminado

**Situação atual**:
* Período disponível na Gold: **02/03/2020 a 14/02/2025** (~5 anos)
* Período **já consultado** em atividades anteriores:
  * Seleção de features
  * Comparação de modelos
  * Tuning de hiperparâmetros
  * Walk-forward validation (até 14/02/2025)

⚠️ **Conclusão**: **NãO existe holdout final intocado no período atual (até 14/02/2025)**

---

### ✅ SOLUÇÃO: Criar Novo Período de Holdout

#### Opção Preferencial

**Período**: **01/03/2025 a 28/02/2026** (12 meses, ~252 pregões)

**Requisitos para viabilidade**:
* ✅ Cotações dos 5 FIIs: Disponíveis até 06/08/2026
* ✅ Dividendos: Disponíveis até 14/08/2026
* ✅ IFIX: Disponível até 05/08/2026
* ⚠️ **SELIC**: Precisa atualizar de 01/01/2025 a 28/02/2026
* ⚠️ **IPCA**: Precisa atualizar de jan/2025 a fev/2026
* ⚠️ **Dólar**: Precisa atualizar de 01/01/2025 a 28/02/2026
* ⚠️ **Desemprego**: Precisa atualizar de jan/2025 a fev/2026
* ✅ **Target calculavel**: Sim, horizonte de 7 pregões está dentro do período disponível de FIIs
* ✅ **Features idênticas**: Sim, mesmas colunas da Gold V1

**Quantidade de observações esperadas**: 
* ~252 pregões × 5 tickers = **~1,260 linhas**
* Para modelo semanal: ~252 ÷ 7 = **~36 previsões semanais × 5 tickers = ~180 observações**

**Status**: ✅ **VIÁVEL** após atualizar variáveis macro

---

#### Opção Alternativa (se dados macro forem limitados)

**Período**: **Últimos 12 meses continuíos após 14/02/2025 com todas as features disponíveis**

Exemplo: Se SELIC/IPCA forem atualizadas apenas até 30/06/2026:
* **Holdout**: 01/03/2025 a 30/06/2026 (~16 meses, ~336 pregões)
* Ainda seria superior ao período mínimo necessário (12 meses)

---

### 📊 Nova Estrutura de Dados Proposta

```
|<-------- Dados já utilizados -------->|<----- Novo Holdout ----->|
02/03/2020                      14/02/2025  01/03/2025     28/02/2026
                                     ^
                                     |
                              Última data da
                              Gold atual
```

**Divisão final sugerida**:

| Período | Datas | Uso | Observações (semanal) |
|----------|-------|-----|---------------------------|
| **Desenvolvimento** | 2020-03 a 2022-12 | Exploração, features, tuning inicial | ~700 |
| **Validação** | 2023-01 a 2024-12 | Walk-forward, ajuste final | ~520 |
| **Holdout Antigo** | 2025-01 a 2025-02 | ⚠️ Já consultado (não usar) | ~50 |
| **Holdout NOVO** | 2025-03 a 2026-02 | ✅ **Teste final único** | ~260 |

---

### 🔒 Garantias de Integridade

**Características do novo holdout**:
1. ✅ **Completamente intocado**: Nunca foi consultado em nenhuma etapa anterior
2. ✅ **Metodologia congelada**: Modelo, features e hiperparâmetros já definidos
3. ✅ **1 execução única**: Resultado aceito ou rejeitado sem iterações
4. ✅ **Dados consistentes**: Mesma estrutura e features da Gold V1
5. ✅ **Horizonte calculavel**: 7 pregões sempre disponíveis dentro do período

---

## 5️⃣ CONGELAMENTO METODOLÓGICO

---

### 🔒 DISCIPLINA DE HOLDOUT: Confirmação

✅ **Confirmado**: Durante a reconstrução e o teste final, o seguinte protocolo será rigorosamente seguido:

---

#### 🚫 O que NÃO SERÁ feito

1. ✖️ **Não criar novas features**
   * Utilizar **apenas** as features existentes na Gold V1
   * Features: `return_7d`, `return_30d`, `return_90d`, `return_180d`, `return_360d`, `volatility_30d`, `volatility_90d`, `dividend_yield_12m`, `days_since_last_dividend`, `alpha_30d`, `alpha_90d`, `selic`, `ipca`, `dolar`, `desemprego`

2. ✖️ **Não realizar nova seleção de features**
   * Conjunto de features já foi validado no notebook `36_feature_selection`
   * Não remover nem adicionar features baseado em desempenho no holdout

3. ✖️ **Não realizar novo tuning de hiperparâmetros**
   * Hiperparâmetros finais já definidos no notebook `38_hyperparameter_tuning`
   * Utilizar **FINAL_HYPERPARAMETERS_REVISED** exatamente como está

4. ✖️ **Não alterar o algoritmo**
   * Modelo: **XGBoost** (XGBClassifier)
   * Não testar outros algoritmos no holdout

5. ✖️ **Não alterar o target**
   * Target: `target_7d` (binário: FII > IFIX em 7 pregões)
   * Target contínuo: `target_alpha_7d` (diferença de retornos)
   * Não alterar horizonte ou metodologia de cálculo

6. ✖️ **Não alterar threshold com base no holdout**
   * Threshold de decisão: 0.5 (default)
   * Não otimizar threshold baseado em resultados do holdout

7. ✖️ **Não redesenhar o modelo**
   * Não usar resultados do holdout para qualquer modificação
   * Não iterar sobre o teste final

---

#### ✅ O que SERÁ feito

1. ✅ **Atualizar apenas os dados brutos**
   * Coletar dados macro de janeiro/2025 a fevereiro/2026
   * Manter exata metodologia de transformação Silver
   * Manter exata metodologia de criação de features Gold

2. ✅ **Reconstruir tabelas com mesma lógica**
   * Silver: Aplicar **mesmos** scripts de transformação
   * Gold: Aplicar **mesmo** código de feature engineering
   * Não alterar nenhuma lógica de cálculo

3. ✅ **Executar o teste final uma única vez**
   * Treinar modelo com hiperparâmetros congelados
   * Avaliar no período 01/03/2025 a 28/02/2026
   * Calcular todas as métricas de desempenho
   * Aceitar ou rejeitar modelo com base nos critérios pré-definidos

4. ✅ **Comparar com meta estabelecida**
   * Meta: Taxa de acerto Top 1 ≈ 58%
   * **Apenas comparar**, não ajustar nada para alcançar a meta
   * Se não atingir, documentar e concluir que modelo não é suficientemente robusto

---

### 📋 Protocolo de Execução

**Ordem das operações**:

```
1. Atualizar dados macro (Silver)
   ↓
2. Reconstruir Gold V1 (até fev/2026)
   ↓
3. Dividir dados:
   - Train/Val: 2020-03 a 2025-02 (período conhecido)
   - Test: 2025-03 a 2026-02 (holdout novo)
   ↓
4. Treinar modelo com hiperparâmetros congelados
   ↓
5. Avaliar no holdout (UMA Única vez)
   ↓
6. Documentar resultados
   ↓
7. Decisão: Aprovar ou Rejeitar
   (SEM iterações)
```

---

### 🔐 Garantia de Integridade

✅ **Confirmado**: Este protocolo garante que:
* O teste final é **verdadeiramente out-of-sample**
* Não haverá **data leakage** do holdout para decisões de modelo
* Resultados representam **desempenho real** esperado em produção
* Não haverá **overfitting** ao conjunto de teste

---

## 6️⃣ PLANO DE RECONSTRUÇÃO DO PIPELINE

---

### 🔄 Ordem de Execução

#### Etapa 1: Atualizar Fontes de Dados Macro

**Ação**: Coletar dados macro de janeiro/2025 a fevereiro/2026

**Fontes**:
1. **SELIC (Taxa Diária)**
   * Fonte: BCB (API ou download manual)
   * Formato: CSV com colunas `data`, `taxa`
   * Período: 01/01/2025 a 28/02/2026
   * Frequência: Diária (dias úteis)

2. **IPCA (Variação Percentual Mensal)**
   * Fonte: IBGE (API Sidra ou download manual)
   * Formato: CSV com colunas `data`, `variacao`
   * Período: Janeiro/2025 a Fevereiro/2026
   * Frequência: Mensal

3. **Dólar PTAX (Cotação de Compra)**
   * Fonte: BCB (API ou download manual)
   * Formato: CSV com colunas `data`, `cotacao`
   * Período: 01/01/2025 a 28/02/2026
   * Frequência: Diária (dias úteis)

4. **Desemprego (PNAD Contínua)**
   * Fonte: IBGE (API Sidra ou download manual)
   * Formato: CSV com colunas `data`, `taxa`
   * Período: Janeiro/2025 a Fevereiro/2026
   * Frequência: Mensal

**Notebooks/Scripts a executar**:
* **Se fontes locais (CSV)**: Atualizar arquivos CSV no diretório de dados
* **Se API**: Verificar se scripts de coleta estão atualizados

---

#### Etapa 2: Atualizar Tabelas Silver

**Ação**: Reprocessar apenas as tabelas Silver de variáveis macro

**Notebooks a executar** (em ordem):

1. 📓 **03_silver_macro_selic** (ou similar)
   * Input: Arquivo CSV ou API BCB
   * Output: `workspace.silver.selic`
   * Transformação: Forward-fill para dias não úteis
   * **Status**: ⚠️ Precisa reexecutar

2. 📓 **03_silver_macro_ipca** (ou similar)
   * Input: Arquivo CSV ou API IBGE
   * Output: `workspace.silver.ipca`
   * Transformação: Replicar valor mensal para todos os dias do mês
   * **Status**: ⚠️ Precisa reexecutar

3. 📓 **03_silver_macro_dolar** (ou similar)
   * Input: Arquivo CSV ou API BCB
   * Output: `workspace.silver.cotacao_dolar`
   * Transformação: Forward-fill para fins de semana/feriados
   * **Status**: ⚠️ Precisa reexecutar

4. 📓 **03_silver_macro_desemprego** (ou similar)
   * Input: Arquivo CSV ou API IBGE
   * Output: `workspace.silver.desemprego`
   * Transformação: Replicar valor mensal para todos os dias do mês
   * **Status**: ⚠️ Precisa reexecutar

**Notebooks a NÃO executar**:
* ✅ Bronze/Silver de FIIs: Já atualizados até agosto/2026
* ✅ Silver IFIX: Já atualizado até agosto/2026

---

#### Etapa 3: Reconstruir Gold V1

**Ação**: Reprocessar tabela Gold com todas as features

**Notebook a executar**:

1. 📕 **04_gold_features_v1** (ou similar)
   * Input: Todas as tabelas Silver
   * Output: `workspace.gold.fii_features_v1`
   * Features: **Mesmas 25 colunas existentes**
   * Targets: `target_7d`, `target_alpha_7d`
   * Período: 2020-03-02 a **2026-02-28** (novo)
   * **Status**: ⚠️ Precisa reexecutar após Silver atualizada

**Validações após reconstruir**:
* Verificar data máxima: Deve ser **2026-02-28**
* Verificar quantidade de linhas: ~6,300 (acréscimo de ~205 linhas)
* Verificar targets não-nulos: Todas as linhas devem ter targets
* Verificar tickers: Todos os 5 tickers até a mesma data

---

#### Etapa 4: Criar Notebook de Teste Final

**Ação**: Criar notebook `40_final_holdout_test`

**Conteúdo**: Ver seção 7️⃣

---

### 📊 Resumo Visual do Fluxo

```
💾 Fontes Externas (BCB/IBGE)
    |
    v
[📑 Coletar dados jan/2025 - fev/2026]
    |
    v
[📓 Atualizar Silver Macro]
  - selic
  - ipca
  - cotacao_dolar
  - desemprego
    |
    v
[📕 Reconstruir Gold V1]
  - fii_features_v1 (até fev/2026)
    |
    v
[📖 Notebook 40_final_holdout_test]
  - Teste final único
  - Aprovar ou Rejeitar
```

---

### ⌛ Estimativa de Esforço

| Etapa | Tempo Estimado | Dificuldade |
|-------|----------------|-------------|
| Coletar dados macro | 1-2 horas | Baixa |
| Atualizar Silver | 30 min | Baixa (reexecutar notebooks) |
| Reconstruir Gold | 15 min | Baixa (reexecutar notebook) |
| Criar notebook teste | 2-3 horas | Média |
| **Total** | **4-6 horas** | **Baixa-Média** |

---

## 7️⃣ DESENHO DO NOTEBOOK `40_final_holdout_test`

---

### 📝 Estrutura do Notebook

O notebook de teste final deverá seguir esta estrutura:

---

#### 📌 Seção 1: Documentação e Objetivo

**Conteúdo**:
* Data de execução
* Objetivo: Teste final do modelo semanal XGBoost
* Período de treino: 2020-03 a 2025-02
* Período de teste: **2025-03 a 2026-02** (holdout intocado)
* Modelo: XGBoost com FINAL_HYPERPARAMETERS_REVISED
* Features: Conjunto validado no notebook 36
* Target: `target_7d` (horizonte de 7 pregões)
* **Garantia**: Este período nunca foi consultado anteriormente

---

#### 📊 Seção 2: Carregar Dados

**Código**:
```python
# Carregar Gold V1 atualizada
df = spark.table("workspace.gold.fii_features_v1")

# Dividir períodos
train_val = df.filter("date <= '2025-02-28'")
holdout = df.filter("date >= '2025-03-01' AND date <= '2026-02-28'")

# Verificar cobertura
print(f"Train/Val: {train_val.count()} linhas")
print(f"Holdout: {holdout.count()} linhas")
print(f"Data mínima holdout: {holdout.agg({'date': 'min'}).collect()[0][0]}")
print(f"Data máxima holdout: {holdout.agg({'date': 'max'}).collect()[0][0]}")
```

**Validações**:
* Holdout deve ter ~1,260 linhas (252 pregões × 5 tickers)
* Todos os 5 tickers devem estar presentes
* Targets devem estar preenchidos para todas as linhas

---

#### 🧠 Seção 3: Treinar Modelo

**Código**:
```python
from xgboost import XGBClassifier
import numpy as np

# Hiperparâmetros congelados (do notebook 38)
FINAL_HYPERPARAMETERS_REVISED = {
    'n_estimators': 100,
    'max_depth': 3,
    'learning_rate': 0.01,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'gamma': 0.1,
    'random_state': 42,
    'eval_metric': 'logloss'
}

# Features selecionadas (do notebook 36)
feature_cols = [
    'return_7d', 'return_30d', 'return_90d', 'return_180d', 'return_360d',
    'volatility_30d', 'volatility_90d',
    'dividend_yield_12m', 'days_since_last_dividend',
    'alpha_30d', 'alpha_90d',
    'selic', 'ipca'
]

# Treinar no período conhecido
X_train = train_val.select(feature_cols).toPandas()
y_train = train_val.select('target_7d').toPandas()['target_7d']

model = XGBClassifier(**FINAL_HYPERPARAMETERS_REVISED)
model.fit(X_train, y_train)

print("✅ Modelo treinado com hiperparâmetros congelados")
```

---

#### 📊 Seção 4: Avaliar no Holdout

**Métricas Clássicas**:

```python
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, 
    recall_score, f1_score, confusion_matrix
)

# Predições no holdout
X_test = holdout.select(feature_cols).toPandas()
y_test = holdout.select('target_7d').toPandas()['target_7d']

y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

# Métricas
roc_auc = roc_auc_score(y_test, y_pred_proba)
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print("\n=== Métricas Clássicas ===")
print(f"ROC-AUC: {roc_auc:.4f}")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")
print(f"\nMatriz de Confusão:\n{cm}")
```

---

#### 🎯 Seção 5: Taxa de Acerto por Contexto

**Por Ticker**:

```python
import pandas as pd

# Adicionar predições ao holdout
holdout_pd = holdout.select('ticker', 'date', 'target_7d').toPandas()
holdout_pd['pred'] = y_pred

# Taxa de acerto por ticker
for ticker in holdout_pd['ticker'].unique():
    subset = holdout_pd[holdout_pd['ticker'] == ticker]
    acc = accuracy_score(subset['target_7d'], subset['pred'])
    print(f"{ticker}: {acc:.2%} ({acc * len(subset):.0f}/{len(subset)} acertos)")
```

**Por Mês**:

```python
holdout_pd['month'] = pd.to_datetime(holdout_pd['date']).dt.to_period('M')

for month in sorted(holdout_pd['month'].unique()):
    subset = holdout_pd[holdout_pd['month'] == month]
    acc = accuracy_score(subset['target_7d'], subset['pred'])
    print(f"{month}: {acc:.2%} ({acc * len(subset):.0f}/{len(subset)} acertos)")
```

**Taxa de Acerto de Previsões Positivas** (quando modelo prevê outperformance):

```python
positive_preds = holdout_pd[holdout_pd['pred'] == 1]
if len(positive_preds) > 0:
    positive_acc = accuracy_score(positive_preds['target_7d'], positive_preds['pred'])
    print(f"\nTaxa de acerto quando prevê outperformance: {positive_acc:.2%}")
    print(f"Total de previsões positivas: {len(positive_preds)}")
```

---

#### 🏆 Seção 6: Estratégia Top 1

**Simular estratégia Top 1**:

```python
# Para cada data única, escolher ticker com maior probabilidade
holdout_pd['prob'] = y_pred_proba

top1_results = []
for date in holdout_pd['date'].unique():
    date_data = holdout_pd[holdout_pd['date'] == date]
    top1_ticker = date_data.nlargest(1, 'prob').iloc[0]
    top1_results.append({
        'date': date,
        'ticker': top1_ticker['ticker'],
        'prob': top1_ticker['prob'],
        'actual': top1_ticker['target_7d']
    })

top1_df = pd.DataFrame(top1_results)
top1_accuracy = top1_df['actual'].mean()

print(f"\n=== Estratégia Top 1 ===")
print(f"Taxa de acerto Top 1: {top1_accuracy:.2%}")
print(f"Total de decisões: {len(top1_df)}")
print(f"Acertos: {top1_df['actual'].sum()}")
```

---

#### 📊 Seção 7: Comparação com Meta

```python
META_TAXA_ACERTO = 0.58

print(f"\n=== Comparação com Meta ===")
print(f"Meta estabelecida: {META_TAXA_ACERTO:.2%}")
print(f"Taxa Top 1 observada: {top1_accuracy:.2%}")
print(f"Diferença: {(top1_accuracy - META_TAXA_ACERTO):.2%}")

if top1_accuracy >= META_TAXA_ACERTO:
    print("✅ Modelo ATINGIU ou SUPEROU a meta")
else:
    print("⚠️ Modelo NÃO atingiu a meta")
    print("   (Isso NÃO significa necessariamente rejeição)")
```

**IMPORTANTE**: 
* A meta de 58% é **referencial**, não um threshold de aprovação/rejeição
* Avaliar também consistência, ROC-AUC, e comparação com benchmark (IFIX)

---

#### 📝 Seção 8: Conclusão e Decisão

**Critérios de Aprovação**:

```markdown
### Decisão Final

Baseado nos resultados:

* ROC-AUC: [valor]
* Taxa Top 1: [valor]
* Consistência por ticker: [análise]
* Consistência temporal: [análise]

**Decisão**: [APROVAR / REJEITAR]

**Justificativa**: [razões baseadas em métricas e consistência]

**Próximos passos**:
* Se aprovado: Implementar em ambiente de produção/monitoramento
* Se rejeitado: Revisar hipóteses fundamentais e coletar mais dados
```

---

## 8️⃣ RECOMENDAÇÃO FINAL E PRÓXIMAS AÇÕES

---

### ❓ RESPOSTAS ÀS PERGUNTAS DA AUDITORIA

#### 1️⃣ Existem dados suficientes hoje para o teste final?

✅ **SIM**, mas com ressalvas:

* **Dados de FIIs e IFIX**: Disponíveis até **agosto de 2026** ✅
* **Dados macro**: Disponíveis apenas até **dezembro de 2024** ⚠️
* **Conclusão**: Dados brutos **existem**, mas variáveis macro precisam ser **atualizadas**

---

#### 2️⃣ Quais tabelas estão desatualizadas?

⚠️ **4 tabelas Silver precisam de atualização**:

| Tabela | Data Atual | Período Necessário | Gap |
|--------|------------|----------------------|-----|
| `silver.selic` | 2024-12-31 | Até 2026-02-28 | **14 meses** |
| `silver.ipca` | 2024-12-01 | Até 2026-02-01 | **15 meses** |
| `silver.cotacao_dolar` | 2024-12-31 | Até 2026-02-28 | **14 meses** |
| `silver.desemprego` | 2024-12-01 | Até 2026-02-01 | **15 meses** |

✅ **Tabelas já atualizadas** (não precisam de ação):
* Bronze/Silver de FIIs: Até agosto/2026
* Silver IFIX: Até agosto/2026

---

#### 3️⃣ Qual período pode ser reservado de forma realmente intocada?

✅ **Período recomendado**: **01/03/2025 a 28/02/2026** (12 meses)

**Garantias**:
* ✅ Nunca foi consultado em nenhuma etapa anterior (desenvolvimento, validação, tuning)
* ✅ Gold atual termina em 14/02/2025 (antes do holdout proposto)
* ✅ Todos os dados necessários estarão disponíveis após atualização
* ✅ ~252 pregões × 5 tickers = ~1,260 observações
* ✅ ~36 decisões semanais × 5 tickers = ~180 observações independentes

**Alternativa** (se dados macro forem limitados):
* Período reduzido até data máxima das variáveis macro disponíveis
* Mínimo aceitável: 8-10 meses (~168-210 pregões)

---

#### 4️⃣ É necessário atualizar scripts locais?

🔹 **DEPENDE da fonte de dados macro**:

**Se usar arquivos CSV locais**:
* ✅ Sim, baixar manualmente dados de SELIC, IPCA, Dólar e Desemprego
* Fontes: Site BCB, IBGE Sidra
* Substituir arquivos CSV existentes com versões atualizadas

**Se usar APIs (BCB/IBGE)**:
* 🔹 Provavelmente não, mas verificar se:
  * Scripts de coleta funcionam corretamente
  * Tokens/credenciais estão válidos
  * Período de coleta está configurado corretamente

**Ação**: Verificar notebooks Silver macro para identificar método usado

---

#### 5️⃣ Qual é a próxima ação concreta?

🎯 **PRÓXIMA AÇÃO IMEDIATA**:

**Etapa 1**: Identificar scripts de injeção Silver
```python
# Executar busca nos notebooks do projeto
# Procurar por: "silver.selic", "silver.ipca", "silver.cotacao_dolar", "silver.desemprego"
# Identificar notebooks responsáveis por criar essas tabelas
```

**Etapa 2**: Coletar dados macro faltantes
* SELIC: Janeiro/2025 a Fevereiro/2026
* IPCA: Janeiro/2025 a Fevereiro/2026
* Dólar: Janeiro/2025 a Fevereiro/2026
* Desemprego: Janeiro/2025 a Fevereiro/2026

**Etapa 3**: Atualizar Silver
* Reexecutar notebooks de injeção Silver para as 4 variáveis macro
* Validar: Verificar data máxima de cada tabela

**Etapa 4**: Reconstruir Gold
* Reexecutar notebook `04_gold_features_v1`
* Validar: Data máxima deve ser ~28/02/2026

**Etapa 5**: Criar notebook teste final
* Criar `40_final_holdout_test` com estrutura da seção 7️⃣
* Executar UMA Única vez
* Documentar resultados

---

#### 6️⃣ Quantas etapas são necessárias até o notebook 40_final_holdout_test?

📊 **5 etapas sequenciais**:

| Etapa | Descrição | Tempo Estimado | Bloqueante? |
|-------|-------------|----------------|-------------|
| **1** | Identificar scripts Silver | 30 min | ⚠️ Sim |
| **2** | Coletar dados macro | 1-2 horas | ⚠️ Sim |
| **3** | Atualizar Silver (4 tabelas) | 30 min | ⚠️ Sim |
| **4** | Reconstruir Gold V1 | 15 min | ⚠️ Sim |
| **5** | Criar e executar notebook 40 | 2-3 horas | — |
| **Total** | | **4-6 horas** | |

**Todas as etapas são bloqueantes** (precisam ser executadas em ordem)

---

### 🚀 PLANO DE AÇÃO RESUMIDO

```
┌─────────────────────────────────────────────┐
│  ETAPA 1: Identificar Scripts [30 min]      │
│  └─ Procurar notebooks Silver macro         │
└─────────────────────────────────────────────┘
         ↓
┌─────────────────────────────────────────────┐
│  ETAPA 2: Coletar Dados [1-2 horas]        │
│  ├─ SELIC (BCB): jan/25-fev/26             │
│  ├─ IPCA (IBGE): jan/25-fev/26             │
│  ├─ Dólar (BCB): jan/25-fev/26             │
│  └─ Desemprego (IBGE): jan/25-fev/26       │
└─────────────────────────────────────────────┘
         ↓
┌─────────────────────────────────────────────┐
│  ETAPA 3: Atualizar Silver [30 min]        │
│  └─ Reexecutar 4 notebooks Silver          │
└─────────────────────────────────────────────┘
         ↓
┌─────────────────────────────────────────────┐
│  ETAPA 4: Reconstruir Gold [15 min]        │
│  └─ Reexecutar notebook Gold V1             │
└─────────────────────────────────────────────┘
         ↓
┌─────────────────────────────────────────────┐
│  ETAPA 5: Teste Final [2-3 horas]          │
│  └─ Criar e executar notebook 40           │
└─────────────────────────────────────────────┘
         ↓
    🎯 DECISÃO: Aprovar/Rejeitar Modelo
```

---

### ✅ CONCLUSÃO DA AUDITORIA

**Resumo Executivo**:

1. ✅ **Dados de FIIs disponíveis até agosto/2026** (suficiente)
2. ⚠️ **Dados macro desatualizados** (param em dezembro/2024)
3. ✅ **Holdout viável**: 01/03/2025 a 28/02/2026 (12 meses intocados)
4. 🔧 **Atualização necessária**: 4 tabelas Silver (SELIC, IPCA, Dólar, Desemprego)
5. ⌛ **Tempo estimado**: 4-6 horas de trabalho
6. 👉 **Próxima ação**: Identificar scripts Silver e coletar dados macro

---

**Status**: ✅ **Teste final é VIÁVEL após atualização de dados macro**

**Risco**: 🟡 **BAIXO** (atualização é simples e direta)

**Próximo Milestone**: 🎯 **Notebook `40_final_holdout_test` pronto para execução**

---

## 📊 RESUMO VISUAL DA AUDITORIA

---

### 🟢 Status Atual das Tabelas

| Camada | Tabela | Data Máxima | Status | Ação Necessária |
|--------|---------|--------------|--------|-------------------|
| **Bronze** | fii_prices | 2026-08-06 | ✅ OK | Nenhuma |
| **Bronze** | fii_dividends | 2026-08-14 | ✅ OK | Nenhuma |
| **Bronze** | ifix | 2026-08-05 | ✅ OK | Nenhuma |
| **Silver** | fii_prices | 2026-08-06 | ✅ OK | Nenhuma |
| **Silver** | fii_dividends | 2026-08-14 | ✅ OK | Nenhuma |
| **Silver** | fii_total_returns | 2026-08-06 | ✅ OK | Nenhuma |
| **Silver** | ifix | 2026-08-05 | ✅ OK | Nenhuma |
| **Silver** | selic | 2024-12-31 | ⚠️ DESATUALIZADA | ⚠️ Atualizar |
| **Silver** | ipca | 2024-12-01 | ⚠️ DESATUALIZADA | ⚠️ Atualizar |
| **Silver** | cotacao_dolar | 2024-12-31 | ⚠️ DESATUALIZADA | ⚠️ Atualizar |
| **Silver** | desemprego | 2024-12-01 | ⚠️ DESATUALIZADA | ⚠️ Atualizar |
| **Gold** | fii_features_v1 | 2025-02-14 | 🟡 LIMITADA | 🔄 Reconstruir após Silver |

---

### 🎯 Checklist de Ações

#### ☐ Etapa 1: Preparar Atualização

* [ ] Identificar notebooks Silver responsáveis pelas variáveis macro
* [ ] Verificar se usam arquivos CSV locais ou APIs externas
* [ ] Preparar fontes de dados (CSV ou verificar credenciais de API)

#### ☐ Etapa 2: Coletar Dados Macro

* [ ] **SELIC**: Baixar taxa diária de 01/01/2025 a 28/02/2026 (BCB)
* [ ] **IPCA**: Baixar variação mensal de jan/2025 a fev/2026 (IBGE)
* [ ] **Dólar PTAX**: Baixar cotação diária de 01/01/2025 a 28/02/2026 (BCB)
* [ ] **Desemprego**: Baixar taxa mensal de jan/2025 a fev/2026 (IBGE)

#### ☐ Etapa 3: Atualizar Silver

* [ ] Executar notebook Silver SELIC
* [ ] Executar notebook Silver IPCA
* [ ] Executar notebook Silver Dólar
* [ ] Executar notebook Silver Desemprego
* [ ] **Validar**: Verificar data máxima de cada tabela = 28/02/2026

#### ☐ Etapa 4: Reconstruir Gold

* [ ] Executar notebook `04_gold_features_v1` (ou equivalente)
* [ ] **Validar**: Data máxima = 2026-02-28
* [ ] **Validar**: Todos os 5 tickers até mesma data
* [ ] **Validar**: Targets calculados para todas as linhas
* [ ] **Validar**: ~6,300 linhas totais (~205 novas linhas)

#### ☐ Etapa 5: Criar Notebook de Teste Final

* [ ] Criar notebook `40_final_holdout_test`
* [ ] Implementar seções 1-8 conforme desenho da seção 7️⃣
* [ ] Garantir que hiperparâmetros estão congelados
* [ ] Garantir que features estão congeladas
* [ ] Garantir que período de holdout = 2025-03-01 a 2026-02-28

#### ☐ Etapa 6: Executar Teste Final

* [ ] **IMPORTANTE**: Executar notebook **UMA Única vez**
* [ ] Calcular todas as métricas (ROC-AUC, Accuracy, Precision, Recall, F1)
* [ ] Calcular taxa de acerto por ticker
* [ ] Calcular taxa de acerto por mês
* [ ] Calcular taxa Top 1
* [ ] Comparar com meta de 58%
* [ ] Documentar resultados

#### ☐ Etapa 7: Decisão Final

* [ ] Analisar consistência dos resultados
* [ ] Avaliar robustez do modelo
* [ ] **Decidir**: Aprovar ou Rejeitar modelo
* [ ] Documentar justificativa da decisão
* [ ] Definir próximos passos (produção ou revisão)

---

### 🚦 Semáforo de Status

```
🟢 BRONZE FIIs/IFIX:  Pronto para uso (até ago/2026)
🟢 SILVER FIIs/IFIX:  Pronto para uso (até ago/2026)
🔴 SILVER MACRO:      Precisa atualizar (gap de 14 meses)
🟡 GOLD V1:           Precisa reconstruir (após Silver)
⚪ HOLDOUT NOVO:      Viável (2025-03 a 2026-02)
🔵 MODELO:            Congelado e pronto para teste
```

---

### 📢 Mensagem Final

👉 **A auditoria está completa. O teste final do modelo semanal é VIÁVEL.**

**Gargalo identificado**: Variáveis macroeconômicas (SELIC, IPCA, Dólar, Desemprego) desatualizadas

**Solução**: Coletar dados macro de jan/2025 a fev/2026 e reconstruir pipeline (4-6 horas)

**Holdout**: Período 01/03/2025 a 28/02/2026 está completamente intocado e pronto para uso

**Próxima ação**: Iniciar Etapa 1 do checklist (identificar scripts Silver)

---

**🚨 LEMBRETE IMPORTANTE**:

⚠️ Durante todo o processo:
* ✖️ NÃO criar novas features
* ✖️ NÃO fazer novo tuning
* ✖️ NÃO alterar o modelo
* ✖️ NÃO iterar sobre o teste final
* ✅ APENAS atualizar dados e executar teste uma vez

---